In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import os
import numpy as np
import json
import rasterio
from tqdm import tqdm
import random
import pickle

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [5]:
BASE_DIR = '/content/drive/MyDrive/flood_detection_project'
RAW_DATA_DIR = BASE_DIR + '/data/raw/SEN12FLOOD/'


In [6]:
S1_JSON_PATH = os.path.join(RAW_DATA_DIR, "S1list.json")

with open(S1_JSON_PATH, "r") as f:
    s1_data = json.load(f)

print(len(s1_data))

335


In [7]:
def log_transform(arr, eps=1e-6):
  return np.log(arr+eps)

def normalize_per_channel (X):
  X_norm = np.zeros_like(X)
  for i in range(X.shape[0]):
    mean = X[i].mean()
    std = X[i].std() + 1e-6
    X_norm[i] = (X[i]-mean)/std
  return X_norm

def center_crop(arr, target_h, target_w):
    h, w = arr.shape
    start_h = (h - target_h) // 2
    start_w = (w - target_w) // 2
    return arr[start_h:start_h+target_h, start_w:start_w+target_w]

def find_vv_vh(event_folder, prefix):
  files = os.listdir(event_folder)

  vv = [f for f in files if f.startswith(prefix) and f.endswith('_VV.tif')]
  vh = [f for f in files if f.startswith(prefix) and f.endswith('_VH.tif')]

  if len(vv) == 0 or len(vh) == 0:
    return None, None

  return vv[0], vh[0]

In [8]:
def build_sample(event_id, event_data, label_type="flood"):
    pre, flood = [], []

    for k, v in event_data.items():
        if k.isdigit():
            if v["FLOODING"]:
                flood.append(v["filename"])
            else:
                pre.append(v["filename"])

    event_folder = os.path.join(RAW_DATA_DIR, event_id)
    if not os.path.exists(event_folder):
        return None

    # Decide which pair to use
    if label_type == "flood":
        if len(pre) == 0 or len(flood) == 0:
            return None
        pre_fname = pre[0]
        post_fname = flood[0]
        y = 1
    else:  # non-flood
        if len(pre) < 2:
            return None
        pre_fname = pre[0]
        post_fname = pre[1]
        y = 0

    pre_vv, pre_vh = find_vv_vh(event_folder, pre_fname)
    post_vv, post_vh = find_vv_vh(event_folder, post_fname)

    if pre_vv is None or post_vv is None:
        return None

    def load_tif(path):
        with rasterio.open(path) as src:
            return src.read(1).astype(np.float32)

    pre_vv_arr   = load_tif(os.path.join(event_folder, pre_vv))
    pre_vh_arr   = load_tif(os.path.join(event_folder, pre_vh))
    post_vv_arr  = load_tif(os.path.join(event_folder, post_vv))
    post_vh_arr  = load_tif(os.path.join(event_folder, post_vh))

    target_h = min(pre_vv_arr.shape[0], post_vv_arr.shape[0])
    target_w = min(pre_vv_arr.shape[1], post_vv_arr.shape[1])

    pre_vv_arr  = center_crop(pre_vv_arr, target_h, target_w)
    pre_vh_arr  = center_crop(pre_vh_arr, target_h, target_w)
    post_vv_arr = center_crop(post_vv_arr, target_h, target_w)
    post_vh_arr = center_crop(post_vh_arr, target_h, target_w)

    X = np.stack([pre_vv_arr, pre_vh_arr, post_vv_arr, post_vh_arr], axis=0)
    X = normalize_per_channel(log_transform(X))

    return X, y


In [9]:
positive_samples = []

for event_id, event_data in tqdm(s1_data.items()):
    sample = build_sample(event_id, event_data, label_type='flood')
    if sample is not None:
        positive_samples.append(sample)

print("Total usable samples:", len(positive_samples))


100%|██████████| 335/335 [05:43<00:00,  1.02s/it]

Total usable samples: 134


In [10]:
negative_samples = []

for event_id, event_data in tqdm(s1_data.items()):
    sample = build_sample(event_id, event_data, label_type='non-flood')
    if sample is not None:
        negative_samples.append(sample)

print("Total usable negative samples:", len(negative_samples))



100%|██████████| 335/335 [10:16<00:00,  1.84s/it]

Total usable negative samples: 286


Balancing the dataset

In [11]:
min_count = min(len(positive_samples), len(negative_samples))
positive_samples = random.sample(positive_samples, min_count)
negative_samples = random.sample(negative_samples, min_count)

dataset = positive_samples + negative_samples
random.shuffle(dataset)

print("Total samples:", len(dataset))
print("Positive samples:", len(positive_samples))
print("Negative samples:", len(negative_samples))

Total samples: 268
Positive samples: 134
Negative samples: 134


In [12]:
n_total = len(dataset)

train_end = int(n_total * 0.7)
val_end = int(n_total * 0.85)

train_data = dataset[:train_end]
val_data = dataset[train_end:val_end]
test_data = dataset[val_end:]

print("train: ", len(train_data))
print("val: ", len(val_data))
print("test: ", len(test_data))
print("total", len(dataset))


train:  187
val:  40
test:  41
total 268


In [13]:
X,y = train_data[0]
print("X shape: ", X.shape)
print("Label: ", y)

X shape:  (4, 516, 544)
Label:  1


In [15]:
SAVE_DIR = '/content/drive/MyDrive/flood_detection_project/data/processed'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(os.path.join(SAVE_DIR, 'train.pkl'), 'wb') as f:
    pickle.dump(train_data, f)

with open(os.path.join(SAVE_DIR, 'val.pkl'), 'wb') as f:
    pickle.dump(val_data, f)

with open(os.path.join(SAVE_DIR, 'test.pkl'), 'wb') as f:
    pickle.dump(test_data, f)

print('datasets saved successfully')

datasets saved successfully
